# DS4DS Exercise Sheet 10

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

## Task 1: Parameter identification of the double compound pendulum 

The goal of this exercise is to perform parameter identification for a non-linear dynamic system. As a system, we consider the double component pendulum ([Source](https://en.wikipedia.org/wiki/Double_pendulum)), which is shown in the following figure.

<img src="./double_pendulum.png" alt="Double pendulum image" width="600"/>

This system consists of two pendulums connected in series, each with its own mass and length (in contrast to the mathematical pendulum where only point masses at the pole tips were assumed!).

In order to obtain the equations of motion and thus the necessary ODEs, we want to utilize Lagragian mechanics ([Source](https://en.wikipedia.org/wiki/Lagrangian_mechanics)). Unlike Newtonian mechanics, which relies on forces and accelerations, Lagrangian mechanics focuses on the concept of energy and utilizes a mathematical function called the Lagrangian. The fundamental principle underlying Lagrangian mechanics is the principle of stationary action. The action $S$ is defined as the integral of the Lagrangian over time, and the path taken by a system between two points in its configuration space is the one for which the action is stationary:

\begin{equation}
S = \int_{t_1}^{t_2} L(q, \dot{q}, t) \mathrm{d}t . \tag{1} 
\end{equation}

where $q$ represents the generalized coordinates of the system, $\dot{q}$ is the corresponding generalized velocity and $t$ is time. The equations of motion are then obtained by applying the Euler-Lagrange equation:

\begin{equation}
\frac{\mathrm{d}}{\mathrm{d}t}\left(\frac{\partial L}{\partial \dot{q}}\right) - \frac{\partial L}{\partial q} = 0 . \tag{2} 
\end{equation}

This equation is derived from the principle of stationary action and provides a concise expression for the dynamics of the system.
The Lagrangian, denoted by $L$, is defined as the difference between the kinetic and potential energies of a system:

\begin{equation}
L=T-V .\tag{3} 
\end{equation}

To specify the kinetic energies $T$ and potential energies $V$ for this system, we need to define the generalized coordinates. Let $\theta_1$ and $\theta_2$ represent the angles of the first and second pendulum joints, respectively. The corresponding generalized coordinates are $q_1 = \theta_1$ and $q_2 = \theta_2$.



In the following, the ODEs of the double pendulum are to be determined. The derivation of the ODEs is carried out step by step, with tests being used to verify the equations determined. 

### **a)** Determining the potential energy of the system

In this system, $V$ is the sum of the potential energies of the two poles.

**Hint 1:** Potential energy measured at the center of gravity $s$ of a body with a mass $m$:
$$
\begin{align}
V = gms
\end{align}
$$

**Hint 2:** Note the geometric arrangement of the system. In the upper equilibrium, the system has maximum potential energy; in the lower equilibrium, the system has negative potential energy.

To calculate the total potential energy $V$ of the system, we need to determine the vertical position (y-coordinate) of the center of gravity for both pendulum links.

Assuming the origin $(0,0)$ is at the top pivot and the y-axis points upwards, the pendulum hanging downwards will reside in negative y-coordinates. For uniform rods, the centers of gravity $s_1$ and $s_2$ are located at the midpoints of their respective lengths ($l_1/2$ and $l_2/2$).

The y-coordinates for the centers of mass are:

* **Link 1:** $y_1 = -s_1 \cos(q_1)$
* **Link 2:** $y_2 = -l_1 \cos(q_1) - s_2 \cos(q_2)$

The total potential energy is the sum of the potential energies of both links:


$$V = m_1 g y_1 + m_2 g y_2$$

In [7]:
import numpy as np

def V(q):
    """
    Computes the potential energy of a two-link pendulum system.

    Args:
        q: Vector of joint angles for the two-link pendulum, shape (2,).

    Returns:
        float: Potential energy of the system.
    """
    g = 9.81
    m_1 = 1.0
    m_2 = 1.0
    l_1 = 2.0
    l_2 = 2.0
    
    # Centers of gravity for uniform rods
    s_1 = l_1 / 2.0
    s_2 = l_2 / 2.0
    
    ### BEGIN SOLUTION
    q1, q2 = q[0], q[1]
    
    # y-coordinates of the centers of mass
    y_1 = -s_1 * np.cos(q1)
    y_2 = -l_1 * np.cos(q1) - s_2 * np.cos(q2)
    
    # Total potential energy
    potential_energy = m_1 * g * y_1 + m_2 * g * y_2
    ### END SOLUTION
    
    return potential_energy

In [8]:
# Evaluation and hard check
q = np.array([0.5, 3.0])
print(f"V: {V(q)}")

assert np.isclose(V(q), -16.1154284047833, atol=1e-5, rtol=0), "Evaluation failed!"

V: -16.1154284047833


### **b)** Determining the kinetic energy of the system

The kinetic energy $T$ of the system is the sum of the translational and rotational energies of the bodies.

**Hint 1:** Kinetic energy of a single body measured at its center of gravity $s = [x_s\;y_s]$:
$$
\begin{align}
T = \frac{1}{2}m|v_s|^2 + \frac{1}{2}J\omega^2,
\end{align}
$$
with $v_s = \frac{\mathrm{d}}{\mathrm{d}t}[x_s\; y_s]^\top$ and $\omega = \frac{\mathrm{d}}{\mathrm{d}t}\theta$.

**Hint 2:** Calculate $v_s$ first and, therefore, represent $[x_s\; y_s]^\top$ by the joints' angles. 

**Hint 3:** Inertia of a pole with mass $m$ and length $l$: $J_{\mathrm{pole}} =\frac{1}{12}ml^2$.


### How the Code Works

The code computes the total kinetic energy $T$ of the system, which consists of both the **translational** energy (the centers of mass moving through space) and the **rotational** energy (the poles spinning around their own centers of mass).

#### 1. Centers of Gravity and Inertia

First, the distances to the centers of gravity ($s_1$ and $s_2$) are defined as the midpoints of the poles. The moments of inertia $J_1$ and $J_2$ are defined using the formula for a uniform thin rod pivoting about its center:


$$J_{\text{pole}} = \frac{1}{12}ml^2$$

#### 2. Deriving Velocity via the Chain Rule

To get the translational kinetic energy, we need the squared velocity vectors $|v_s|^2$. The position of the center of mass for the second link ($x_{s2}$, $y_{s2}$) is found by adding the vector of the entire first link to the vector of the bottom half of the second link:


$$x_{s2} = l_1 \sin(q_1) + s_2 \sin(q_2)$$

$$y_{s2} = -l_1 \cos(q_1) - s_2 \cos(q_2)$$

We take the time derivative of these positions to find the velocities ($\dot{x}_{s2}$ and $\dot{y}_{s2}$). Because the angles $q$ change with time, we must use the chain rule, which pulls out the angular velocities $\dot{q}$ (represented in code as `dq1` and `dq2`):


$$\dot{x}_{s2} = l_1 \cos(q_1)\dot{q}_1 + s_2 \cos(q_2)\dot{q}_2$$

$$\dot{y}_{s2} = l_1 \sin(q_1)\dot{q}_1 + s_2 \sin(q_2)\dot{q}_2$$

#### 3. Summing the Energies

Once the $x$ and $y$ velocity components are derived, squaring and adding them yields the squared magnitude of the velocity ($|v_s|^2 = \dot{x}^2 + \dot{y}^2$).

Finally, the function applies the standard physics formulas provided in the hint to sum up the partial energies into a single, total kinetic energy scalar:


$$T = \left(\frac{1}{2}m_1|v_{s1}|^2 + \frac{1}{2}J_1\dot{q}_1^2\right) + \left(\frac{1}{2}m_2|v_{s2}|^2 + \frac{1}{2}J_2\dot{q}_2^2\right)$$

In [9]:
import numpy as np

def T(q, q_p):
    """
    Computes the kinetic energy of a two-link pendulum system.

    Args:
        q: Vector of joint angles for the two-link pendulum, shape (2,).
        q_p: Vector of joint angular velocities, shape (2,).

    Returns:
        float: Kinetic energy of the system.
    """
    m_1 = 1.0
    m_2 = 1.0
    l_1 = 2.0
    l_2 = 2.0
    
    # Centers of gravity (midpoints of the poles)
    s_1 = l_1 / 2.0
    s_2 = l_2 / 2.0
    
    # Moments of inertia for poles rotating around their center of mass
    J_1 = (1.0 / 12.0) * m_1 * l_1**2
    J_2 = (1.0 / 12.0) * m_2 * l_2**2
    
    ### BEGIN SOLUTION
    q1, q2 = q[0], q[1]
    dq1, dq2 = q_p[0], q_p[1]
    
    # 1. Kinematics for Link 1
    # Velocity components of the center of mass for Link 1
    dx_s1 = s_1 * np.cos(q1) * dq1
    dy_s1 = s_1 * np.sin(q1) * dq1
    v_s1_sq = dx_s1**2 + dy_s1**2  # Squared velocity magnitude |v_s1|^2
    
    # 2. Kinematics for Link 2
    # Velocity components of the center of mass for Link 2
    dx_s2 = l_1 * np.cos(q1) * dq1 + s_2 * np.cos(q2) * dq2
    dy_s2 = l_1 * np.sin(q1) * dq1 + s_2 * np.sin(q2) * dq2
    v_s2_sq = dx_s2**2 + dy_s2**2  # Squared velocity magnitude |v_s2|^2
    
    # 3. Kinetic Energy Calculation
    # Translational kinetic energy (1/2 * m * v^2)
    T_trans_1 = 0.5 * m_1 * v_s1_sq
    T_trans_2 = 0.5 * m_2 * v_s2_sq
    
    # Rotational kinetic energy (1/2 * J * w^2)
    T_rot_1 = 0.5 * J_1 * dq1**2
    T_rot_2 = 0.5 * J_2 * dq2**2
    
    # Total kinetic energy
    kinetic_energy = T_trans_1 + T_rot_1 + T_trans_2 + T_rot_2
    ### END SOLUTION
    
    return kinetic_energy

In [10]:
q = [0.5, 3.0]
q_p = [0.4, -2]

print(f"T: {T(q, q_p)}")

assert np.isclose(T(q, q_p), 4.375163118208428, atol=1e-5, rtol=0), "Evaluation failed!"

T: 4.375163118208428


Inserting eq. (3) into eq. (2) delivers:

\begin{equation}
\frac{\partial}{\partial \dot{q}}\left(\frac{\partial T}{\partial \dot{q}}\right)^\top \ddot{q} - \left( -\frac{\partial}{\partial q} \left(\frac{\partial T}{\partial \dot{q}}\right)^\top \dot{q} - \frac{\partial}{\partial t} \left(\frac{\partial T}{\partial \dot{q}}\right)^\top + \frac{\partial T}{\partial q}^\top - \left(\frac{\partial V}{\partial q}\right)^\top \right) = 0. \tag{4}
\end{equation}

We define

\begin{equation}
M(t,q) = \frac{\partial}{\partial \dot{q}}\left(\frac{\partial T}{\partial \dot{q}}\right)^\top
\end{equation}

as the mass matrix and

\begin{equation}
h(t,q,\dot{q}) =  \left( -\frac{\partial}{\partial q} \left(\frac{\partial T}{\partial \dot{q}}\right)^\top \dot{q} - \frac{\partial}{\partial t} \left(\frac{\partial T}{\partial \dot{q}}\right)^\top + \frac{\partial T}{\partial q}^\top - \left(\frac{\partial V}{\partial q}\right)^\top \right)
\end{equation}

the h-vector, which summarizes all gyroscopic and active forces.
We can therefore write the equation again in simplified form:

\begin{equation}
M(t,q) \ddot{q} - h(t,q,\dot{q}) =  0 . \tag{5}
\end{equation}

### **c)** Calculation $M(t,q)$ and $h(t,q,\dot{q})$

**Hint:** Here you should calculate the derivatives by hand based on your results for $T$ and $V$ from the previous subtasks.

### Deriving the Mass Matrix Elements

We extract it directly from the kinetic energy equation set up in the previous step.

The kinetic energy of any mechanical system can be written in a compact quadratic form using the mass matrix:


$$T = \frac{1}{2} \dot{q}^\top M(q) \dot{q}$$

For a two-degree-of-freedom system, expanding this matrix multiplication looks like this:


$$T = \frac{1}{2} \left( M_{11}\dot{q}_1^2 + 2M_{12}\dot{q}_1\dot{q}_2 + M_{22}\dot{q}_2^2 \right)$$

If we expand the squared velocities from your previous kinetic energy function and group the terms by $\dot{q}_1^2$, $\dot{q}_2^2$, and $\dot{q}_1\dot{q}_2$, we can map them directly to the matrix elements:

1. **$M_{11}$ (Inertia acting on Link 1):** This gathers all terms multiplied by $\dot{q}_1^2$. It includes the rotational inertia of the first link ($J_1$), the translational inertia of its center of mass ($m_1 s_1^2$), and the translational inertia of the second link's mass acting at the end of the first link ($m_2 l_1^2$).
2. **$M_{22}$ (Inertia acting on Link 2):** This gathers the terms multiplied by $\dot{q}_2^2$. It only includes the rotational inertia of the second link ($J_2$) and the translational inertia of its own center of mass ($m_2 s_2^2$).
3. **$M_{12}$ and $M_{21}$ (Coupled Inertia):** This gathers the terms multiplied by $\dot{q}_1\dot{q}_2$. This represents how accelerating one link affects the other. Because $\dot{x}_{s2}$ and $\dot{y}_{s2}$ contain trigonometric terms that were squared and added, trigonometric identities reduce the coupling element to $m_2 l_1 s_2 \cos(q_1 - q_2)$.

In [11]:
def M(q, m_1, m_2, l_1, l_2):
    """
    Computes the mass matrix for a two-link pendulum system.

    Args:
        q: Vector of joint angles for the two-link pendulum, shape (2,).
        m_1: Mass of the first pendulum link.
        m_2: Mass of the second pendulum link.
        l_1: Length of the first pendulum link.
        l_2: Length of the second pendulum link.

    Returns:
        np.ndarray: Mass matrix of the system, shape (2, 2).
    """
    g = 9.81
    
    q = np.asarray(q)
    assert q.shape == (2,), "q must be of shape (2,)"
    
    ### BEGIN SOLUTION
    q1, q2 = q[0], q[1]
    
    # Centers of gravity (midpoints of the poles)
    s_1 = l_1 / 2.0
    s_2 = l_2 / 2.0
    
    # Moments of inertia for poles
    J_1 = (1.0 / 12.0) * m_1 * l_1**2
    J_2 = (1.0 / 12.0) * m_2 * l_2**2
    
    # Calculate mass matrix elements
    M11 = m_1 * s_1**2 + J_1 + m_2 * l_1**2
    M22 = m_2 * s_2**2 + J_2
    M12 = m_2 * l_1 * s_2 * np.cos(q1 - q2)
    M21 = M12  # The mass matrix is always symmetric
    
    mass_matrix = np.array([
        [M11, M12],
        [M21, M22]
    ])
    ### END SOLUTION
    
    assert mass_matrix.shape == (2, 2), "Mass matrix must be 2x2"
    return mass_matrix

In [12]:
q = np.array([0.5, 3.0])

m_1 = 1.0
m_2 = 1.0
l_1 = 2.0
l_2 = 2.0

# Calculate the mass matrix
M_result = M(q, m_1, m_2, l_1, l_2)

# Print the result using f-strings
print(f"M: {M_result}")

# Define the expected matrix
expected_M = np.array([
    [5.333333333333333, -1.6022872310938674],
    [-1.6022872310938674, 1.333333333333333]
])

# Assert using numpy.allclose for element-wise tolerance checking
assert np.allclose(M_result, expected_M, atol=1e-5, rtol=0), "Matrix M does not match expected values!"

M: [[ 5.33333333 -1.60228723]
 [-1.60228723  1.33333333]]


In [13]:
def h(q, q_p, m_1, m_2, l_1, l_2):
    """
    Computes the h-vector for a two-link pendulum system.

    Args:
        q: Vector of joint angles for the two-link pendulum, shape (2,).
        q_p: Vector of joint angular velocities, shape (2,).
        m_1: Mass of the first pendulum link.
        m_2: Mass of the second pendulum link.
        l_1: Length of the first pendulum link.
        l_2: Length of the second pendulum link.

    Returns:
        np.ndarray: h-vector of the system, shape (2,).
    """
    g = 9.81
    
    q = np.asarray(q)
    q_p = np.asarray(q_p)
    assert q.shape == (2,), "q must be of shape (2,)"
    assert q_p.shape == (2,), "q_p must be of shape (2,)"
    
    ### BEGIN SOLUTION
    q1, q2 = q[0], q[1]
    dq1, dq2 = q_p[0], q_p[1]
    
    # Centers of gravity (midpoints of the poles)
    s_1 = l_1 / 2.0
    s_2 = l_2 / 2.0
    
    # 1. Centrifugal / Coriolis Terms
    # These emerge from the chain-rule expansion of the kinetic energy derivatives.
    # The common coupling factor is m_2 * l_1 * s_2
    k = m_2 * l_1 * s_2
    c_1 = -k * np.sin(q1 - q2) * dq2**2
    c_2 =  k * np.sin(q1 - q2) * dq1**2
    
    # 2. Gravity Terms (-dV/dq)
    # These are the partial derivatives of the potential energy derived earlier, 
    # negated as per the h-vector definition.
    g_1 = -(m_1 * s_1 + m_2 * l_1) * g * np.sin(q1)
    g_2 = -m_2 * s_2 * g * np.sin(q2)
    
    # 3. Assemble the h-vector
    h_1 = c_1 + g_1
    h_2 = c_2 + g_2
    
    h_vector = np.array([h_1, h_2])
    ### END SOLUTION
    
    assert h_vector.shape == (2,), "h_vector must be of shape (2,)"
    return h_vector

### The Physics Behind the Code

As established in our previous mathematical definitions, the $h$-vector captures all forces acting on the joints that are **independent of the current angular acceleration**. This is split into two physical phenomena:

1. **Centrifugal Forces (`c_1`, `c_2`):** These non-linear terms arise because the links are rotating.
* Notice that `c_1` only depends on the squared velocity of the second link ($\dot{q}_2^2$). This means the centrifugal force of the bottom link spinning pulls on the top joint.
* Conversely, `c_2` depends on ($\dot{q}_1^2$), meaning the top link's rotation exerts a fictitious force on the bottom joint.


2. **Gravitational Forces (`g_1`, `g_2`):** These terms represent the static pull of gravity attempting to pull both links downward to their stable equilibrium at $q = [0, 0]^\top$. This is simply the negative gradient of the potential energy function $V(q)$ you established earlier.

In [14]:
q = [0.5, 3.0]
q_p = [-1.0, 0.2]

m_1 = 1
m_2 = 1
l_1 = 2
l_2 = 2

print("h: $(h(q,q_p, m_1, m_2, l_1, l_2))")

assert np.allclose(h(q, q_p, m_1, m_2, l_1, l_2), [-14.061615829593379, -2.5813315672752104], atol=1e-10, rtol=0) # evaluation at a single point as help for students for hard check 

h: $(h(q,q_p, m_1, m_2, l_1, l_2))


By rearranging eq. (5), we obtain

\begin{equation}
\ddot{q} =  M(t,q)^{-1}h(t,q,\dot{q}) \tag{6},
\end{equation}

which is now to be converted into the state-space form in the last step.
The following state vector is introduced for this purpose:

\begin{equation}
x =  [\theta_1\;\theta_2\;\dot{\theta_1}\;\dot{\theta_2}]^\top.
\end{equation}

Which in turn results in the following state-space equation:

\begin{equation}
\dot{x} =  [\dot{q}\;\ddot{q}]^\top = [\dot{q}\;M(t,q)^{-1}h(t,q,\dot{q})]^\top = f(x,t).
\end{equation}

### The Concept: State-Space Representation

The Euler-Lagrange equations yielded a system of two **second-order** ordinary differential equations (ODEs), $\ddot{q}_1$ and $\ddot{q}_2$.

However, almost all standard numerical integration algorithms—such as the Runge-Kutta 4 methods you often use for forward simulation—are designed to solve **first-order** ODEs.

The state-space representation is a mathematical trick to bridge this gap. By defining a new "state vector" $x$ that stacks both the positions ($q$) and their velocities ($\dot{q}$) together:


$$x = \begin{bmatrix} \theta_1 \\ \theta_2 \\ \dot{\theta}_1 \\ \dot{\theta}_2 \end{bmatrix}$$

We can rewrite the $N$ second-order equations as $2N$ first-order equations. The derivative of this state vector, $\dot{x}$, simply contains the velocities (which we already have in $x$) and the accelerations (which we calculate using the mass matrix and h-vector).

Getting the system into this standard $\dot{x} = f(x, t)$ form is the final preparatory step before passing the dynamics into an ODE solver or discretizing it to build the internal prediction model for an MPC controller.

### **d)** Convert the equation of motion to the state-space representation 

In [15]:
def double_pendulum(dx, x, p, t):
    """
    Computes the derivatives of the state vector `x` for a double pendulum system.

    Args:
        dx: Vector to store the computed derivatives, shape (4,).
        x: State vector containing the generalized coordinates and their derivatives.
            - x[0]: Angle theta_1 (first pendulum)
            - x[1]: Angle theta_2 (second pendulum)
            - x[2]: Angular velocity omega_1 (first pendulum)
            - x[3]: Angular velocity omega_2 (second pendulum)
        p: Parameters tuple/list containing the lengths of the pendulum arms.
            - p[0]: Length of the first pendulum arm (l_1)
            - p[1]: Length of the second pendulum arm (l_2)
        t: Current time (not used in the function, required by ODE solvers).

    Returns:
        None (dx is modified in-place)
    """
    g = 9.81
    m_1 = 1.0
    m_2 = 0.5  # Note the parameter update from the prompt
    
    l_1, l_2 = p
    
    x = np.asarray(x)
    assert x.shape == (4,), "State vector x must have exactly 4 elements."
    assert len(p) == 2, "Parameters vector p must have exactly 2 elements."
    
    # Generalized coordinates and velocities
    q = x[0:2]
    q_p = x[2:4]
    
    ### BEGIN SOLUTION
    # 1. Calculate the Mass matrix and h-vector at the current state
    M_matrix = M(q, m_1, m_2, l_1, l_2)
    h_vector = h(q, q_p, m_1, m_2, l_1, l_2)
    
    # 2. Solve for angular accelerations (q_ddot = M^-1 * h)
    # np.linalg.solve is numerically more stable than explicitly inverting M
    q_ddot = np.linalg.solve(M_matrix, h_vector)
    
    # 3. Populate the derivative vector dx
    # The derivative of position is velocity
    dx[0] = q_p[0]
    dx[1] = q_p[1]
    
    # The derivative of velocity is acceleration
    dx[2] = q_ddot[0]
    dx[3] = q_ddot[1]
    ### END SOLUTION

    return dx # Returning dx is optional for strictly in-place, but standard for scipy compatibility

### 1. The Function Signature (`dx, x, p, t`)

* **`dx` (Derivative of State):** This is the array we need to fill. It represents $\dot{x}$ (how fast the state is changing right now).
* **`x` (Current State):** The current exact position and speed of the pendulum.
* **`p` (Parameters):** Constant physical traits of the system (in this case, lengths $l_1$ and $l_2$).
* **`t` (Time):** The current simulation time. It isn't used in the math here because a standard pendulum's physics don't change based on what time of day it is (it is a time-invariant system), but the solver still provides it.

### 2. Unpacking the State

```python
q = x[0:2]
q_p = x[2:4]

```

The state vector $x$ contains four numbers. We split it in half. `q` gets the two joint angles ($\theta_1, \theta_2$), and `q_p` gets the two angular velocities ($\dot{\theta}_1, \dot{\theta}_2$).

### 3. Calculating the Dynamics Matrices

```python
M_matrix = M(q, m_1, m_2, l_1, l_2)
h_vector = h(q, q_p, m_1, m_2, l_1, l_2)

```

We pass the current angles and speeds into the helper functions derived earlier.

* `M_matrix` tells us how much rotational inertia the system has *in this exact posture*.
* `h_vector` calculates the immediate gravitational pull and the centrifugal/Coriolis forces caused by the current spinning speed.

### 4. Solving for Acceleration

```python
q_ddot = np.linalg.solve(M_matrix, h_vector)

```

This is the most computationally important line. We need the angular accelerations ($\ddot{q}$). From the manipulator equation $M\ddot{q} - h = 0$, we know that $M\ddot{q} = h$.

While mathematically this means $\ddot{q} = M^{-1}h$, explicitly calculating the inverse of a matrix is computationally expensive and can introduce floating-point errors. `np.linalg.solve` is an optimized algorithm that solves the system of linear equations directly, which is the standard best practice in scientific computing.

### 5. Packing the Results

```python
dx[0] = q_p[0]
dx[1] = q_p[1]
dx[2] = q_ddot[0]
dx[3] = q_ddot[1]

```

Finally, we assemble the derivative vector $\dot{x}$ to hand back to the solver:

* The rate of change of the angles `dx[0:2]` is simply the current velocities `q_p`.
* The rate of change of the velocities `dx[2:4]` is the newly calculated accelerations `q_ddot`.

By modifying `dx` in place, the ODE solver can update the system's state for the next time step.

In [16]:
from scipy.integrate import solve_ivp

# SciPy requires the signature fun(t, y, *args) and expects an array returned,
# so we create a small wrapper around your in-place function.
def double_pendulum_scipy(t, x, l1, l2):
    dx = np.zeros_like(x)
    p = [l1, l2]
    double_pendulum(dx, x, p, t)
    return dx

# 1. Setup Initial Conditions and Parameters
x0 = np.array([np.pi / 2, np.pi / 2, 0.0, 0.0])
p = [1.0, 1.0]
tspan = (0.0, 0.1)
dt = 1e-3

# Generate the specific time points we want to save (equivalent to saveat=dt)
t_eval = np.arange(tspan[0], tspan[1] + dt, dt)

# 2. Solve the ODE
# RK45 is the standard explicit Runge-Kutta method in SciPy
sol = solve_ivp(
    fun=double_pendulum_scipy, 
    t_span=tspan, 
    y0=x0, 
    method='RK45', 
    t_eval=t_eval, 
    args=(p[0], p[1])
)

# 3. Extract Results
t = sol.t
x = sol.y  # Already in the shape (4, N), acting like Julia's reduced hcat

# 4. Verify Final State
expected_final_state = np.array([
    1.5039951673202, 
    1.5973123079743048, 
    -1.3326403676290886, 
    0.5208390338127351
])

# Compare the last column of the state matrix x[:, -1] against the expected values
assert np.allclose(x[:, -1], expected_final_state, atol=1e-3, rtol=0), "Simulation final state does not match!"
print("Simulation completed and verified successfully.")

Simulation completed and verified successfully.


First, we generate our ground truth data, for example from a test bench of the system to be idendified. We assume that there only additive, uncorrelated and normally distributed measurement noise.

In [17]:
def gen_measurement(ode_function, x0, tspan, dt, sigma):
    """
    Generates measurement data for a simulated system trajectory.

    Args:
        ode_function: Function representing the ODEs of the system dynamics.
                      Must have the signature `fun(t, y, *args)` for scipy.
        x0: Initial state vector of the system, shape (4,).
        tspan: Tuple (t_start, t_end) specifying the time frame for simulation.
        dt: Time step for saving the simulation results.
        sigma: Standard deviation of the measurement noise.

    Returns:
        tuple: The resulting trajectory `x` and the noisy measurements `y`.
        - x: State trajectory matrix of size (n_states, n_timepoints).
        - y: Noisy measurements matrix of size (n_measurement_states, n_timepoints).
    """
    p_true = (0.5, 2.0)  # ground truth parameters
    
    x0 = np.asarray(x0)
    assert x0.shape == (4,), "Initial state x0 must have shape (4,)"
    
    # Define the exact time points to save the results (equivalent to saveat=dt)
    t_eval = np.arange(tspan[0], tspan[1] + dt, dt)
    
    # Solve the ODE using RK45 (equivalent to Tsit5)
    # The true parameters are passed via the `args` tuple
    sol = solve_ivp(
        fun=ode_function,
        t_span=tspan,
        y0=x0,
        method='RK45',
        t_eval=t_eval,
        args=p_true
    )
    
    t = sol.t
    x = sol.y  # sol.y is already formulated as a matrix, skipping reduce(hcat, x)
    
    assert x.shape == (4, len(t)), "Trajectory matrix shape mismatch"
    
    # Define and add measurement noise
    # R represents the noise covariance matrix
    R = (sigma**2) * np.eye(2)
    
    # Generate random normal noise and add it to the first two states (theta_1, theta_2)
    # Note: Julia's 1-based indexing `x[1:2, :]` becomes 0-based `x[0:2, :]` in Python
    # We use the `@` operator for matrix multiplication equivalent to Julia's `*`
    y = x[0:2, :] + R @ np.random.randn(2, x.shape[1])
    
    return x, y

## Task 2 - Implementation of a function that simulates an experiment 

In order to estimate the parameters, a function is to be added which carries out a whole simulation with the estimated parameters. Use a standard ODE solver `Tsit5()` to simulate a state trajecory based on an arbitrary initial state.

In [18]:
import numpy as np
from scipy.integrate import solve_ivp

def sim_exp(ode_function, x0, tspan, dt, p):
    """
    Generates simulation data for a given system and simulation parameters.

    Args:
        ode_function: Function representing the ODEs of the system dynamics.
                      Must have the signature `fun(t, y, *args)` for scipy.
        x0: Initial state vector of the system, shape (4,).
        tspan: Tuple (t_start, t_end) specifying the time frame for simulation.
        dt: Time step for saving the simulation results.
        p: Parameter iterable (e.g., list or tuple) containing system-specific parameters.

    Returns:
        tuple: The resulting trajectory `x` and the simulated measurements `y`.
        - x: State trajectory matrix of size (n_states, n_timepoints).
        - y: Simulated measurements matrix of size (n_measurement_states, n_timepoints).
    """
    x0 = np.asarray(x0)
    assert x0.shape == (4,), "Initial state x0 must have shape (4,)"
    
    ### BEGIN SOLUTION
    # Define the exact time points to save the results (equivalent to saveat=dt in Julia)
    # Adding a tiny buffer to the end ensures the final time point is included despite float precision
    t_eval = np.arange(tspan[0], tspan[1] + (dt / 2), dt)
    
    # Solve the ODE using RK45 (SciPy's equivalent to Tsit5)
    sol = solve_ivp(
        fun=ode_function,
        t_span=tspan,
        y0=x0,
        method='RK45',
        t_eval=t_eval,
        args=p  # Pass the parameter tuple directly to the solver
    )
    
    t = sol.t
    x = sol.y  # SciPy automatically returns the trajectory as a 2D array
    ### END SOLUTION
    
    assert x.shape == (4, len(t)), "Trajectory matrix shape mismatch"
    
    # Extract the simulated measurements (the first two rows representing the angles)
    # Julia's 1-based x[1:2, :] becomes Python's 0-based exclusive-end x[0:2, :]
    y = x[0:2, :]
    
    return x, y

In [19]:
# 1. Setup Initial Conditions and Parameters
x0 = np.array([np.pi / 2, np.pi / 2, 0.0, 0.0])
tspan = (0.0, 0.1)
dt = 1e-3
p = [1.0, 1.0]

# 2. Run the experiment simulation
x_hat, y_hat = sim_exp(double_pendulum_scipy, x0, tspan, dt, p)

# 3. Verify Output Dimensions
# 0.1 seconds / 0.001 dt = 100 steps + 1 initial condition = 101 time points
assert y_hat.shape == (2, 101), f"Expected shape (2, 101), got {y_hat.shape}"

# 4. Print and Verify Final State
# Python uses -1 to access the last element (equivalent to 'end' in Julia)
print(f"y_hat: {y_hat[:, -1]}")

expected_y_hat_end = np.array([1.5039951673202, 1.5973123079743048])
assert np.allclose(y_hat[:, -1], expected_y_hat_end, atol=1e-5, rtol=0), "Final measurement does not match expected values!"

print("Test passed successfully.")

y_hat: [1.50399544 1.59731156]
Test passed successfully.


## Task 3: Solve an identification problem for unknown pendulum length values 

We assume that we have ideal structural knowledge regarding the double pendulum dynamics, but lack the length information of the two poles. Hence, we want to set up an optimization problem, which identifies the poles' length. First, a cost function must be defined for this purpose.

### **a)** Write a function to compute the MSE. 

In [24]:
import numpy as np

def mse(y, y_hat):
    """
    Computes the Mean Squared Error (MSE).
    Sums the errors across the states (axis 0), and averages over time (axis 1).
    """
    y = np.asarray(y)
    y_hat = np.asarray(y_hat)
    
    # 1. Squared differences
    sq_err = (y - y_hat) ** 2
    
    # 2. Sum across states (rows), then mean across time (columns)
    mean_squared_error = np.mean(np.sum(sq_err, axis=0))
    
    return float(mean_squared_error)

In [25]:
# 1. Create the test matrices
# Note: np.ones takes a tuple for the shape in Python
y = np.ones((2, 5))
y_hat = 3 * np.ones((2, 5))

# 2. Calculate MSE
l = mse(y, y_hat)

# 3. Verify it evaluates to a scalar
# In Julia, length(scalar) == 1. In Python, floats don't have a length, 
# so we assert the type is a float instead.
assert isinstance(l, float), "Expected l to be a scalar float"

# 4. Print and verify the value
print(f"l: {l}")

# Assert using np.isclose for floating point tolerance
assert np.isclose(l, 8.0, atol=1e-5, rtol=0), f"Expected 8.0, but got {l}"
print("Test passed successfully.")

l: 8.0
Test passed successfully.


### **b)** Set up the objective function for the simulation case 

In [26]:
def simulation_objective(p, y_mes, ode_func, x0, tspan, dt):
    """
    Computes the MSE loss value for a simulation-based optimization problem.

    Args:
        p: Parameter vector to be optimized (e.g., [l_1, l_2]).
        y_mes: Matrix of measured data to compare against simulation results.
        ode_func: Function representing the ODEs of the system dynamics.
        x0: Initial state vector of the system.
        tspan: Tuple (t_start, t_end) specifying the time frame for simulation.
        dt: Time step for saving the simulation results.

    Returns:
        float: Objective value representing the mean squared error between 
               measured and simulated data.
    """
    ### BEGIN SOLUTION
    # 1. Run the forward simulation with the optimizer's current guess for `p`
    # We use '_' to discard the full state trajectory 'x' and only keep the measurements 'y_hat'
    _, y_hat = sim_exp(ode_func, x0, tspan, dt, p)
    
    # 2. Calculate the Mean Squared Error between the simulation and ground truth
    loss = mse(y_mes, y_hat)
    ### END SOLUTION
    
    return loss

In [27]:
# Please leave this cell as its is
### BEGIN TESTS

test_params = [[1, 2], [0.5, 2.0], [2, 10], [1.0, 1.0]]
expected_losses = [0.00012865911601969663, 0.0, 0.00038024159885250943, 0.00025941058147801224]

# Using np.pi for pi
x0 = np.array([np.pi / 5, -np.pi / 2, 0.0, 0.0])
tspan = (0.0, 0.1)
dt = 1e-3

# Generate ground truth measurements with 0 noise (sigma = 0)
# Note: Using the double_pendulum_scipy wrapper from earlier steps
_, y_mes = gen_measurement(double_pendulum_scipy, x0, tspan, dt, 0.0)

for p, expected_loss in zip(test_params, expected_losses):
    # Explicitly passing all required arguments rather than relying on global defaults
    given_loss = simulation_objective(p, y_mes, double_pendulum_scipy, x0, tspan, dt)
    
    print(f"Given loss: {given_loss}")
    print(f"Expected loss: {expected_loss}")
    
    assert np.isclose(given_loss, expected_loss, rtol=0, atol=1e-5), f"Assertion failed for p={p}"

print("\nAll tests passed successfully.")
### END TESTS

Given loss: 0.00012866080017554215
Expected loss: 0.00012865911601969663
Given loss: 0.0
Expected loss: 0.0
Given loss: 0.0003802423172988942
Expected loss: 0.00038024159885250943
Given loss: 0.00025941170178233296
Expected loss: 0.00025941058147801224

All tests passed successfully.


Based on identification framework, we utilize the Optim.jl package to solve for the pendulum length. This is provided as a executation-ready code, which you can play around with in the following. As we have generated the ground truth data using the above simulation ourselves, you can compare the found results for the length values and check if you have implemented everything correctly. This last step of configuring and executing the optimization serves as a baseline for the next task. 

In [ ]:
from scipy.optimize import minimize

# 1. Initialization
p_init = np.array([0.7, 1.7])  # initial guess of length values
x0 = np.array([np.pi / 2, 0.0, 0.0, 0.0])
tspan = (0.0, 2.0)
dt = 1e-3

# 2. Measurement Configuration
sigma = 0.1  # measurement noise config

# Generate the noisy ground truth measurements
_, y_mes = gen_measurement(double_pendulum_scipy, x0, tspan, dt, sigma)

# 3. Optimization Setup
# SciPy's 'BFGS' is a standard, highly efficient gradient-based solver.
solver_method = 'BFGS'    
tolerance = 1e-5  # equivalent to x_tol

# Define an inline lambda function to map the parameters correctly for SciPy
objective_func = lambda p: simulation_objective(p, y_mes, double_pendulum_scipy, x0, tspan, dt)

# 4. Execute Optimization
result = minimize(
    fun=objective_func,
    x0=p_init,
    method=solver_method,
    tol=tolerance
)

# Print the full convergence report
print("--- Optimization Report ---")
print(result)

# Extract and print the final optimal parameters
optimal_parameters = result.x
print(f"\nOptimal Parameters: {optimal_parameters}")

# Assert the dimensions of the result
assert optimal_parameters.shape == (2,), "Optimal parameters must have shape (2,)"

--- Optimization Report ---
  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 0.00019813124160144056
        x: [ 5.000e-01  2.001e+00]
      nit: 8
      jac: [-3.587e-07 -4.190e-08]
 hess_inv: [[ 2.234e-02 -1.183e-01]
            [-1.183e-01  3.883e+00]]
     nfev: 33
     njev: 11

Optimal Parameters: [0.49998913 2.00058416]


**Hint:** Try changing the variance of the measurement noise and also try other solvers. Gradient descent works well for convex problems, but if the cost landscape is very nonlinear, it needs a good initialization, therefore global algorithms are better suited for problems where you don't have a good initial guess. The `Optim.jl` package comes with various solvers which you can find [here](https://julianlsolvers.github.io/Optim.jl/stable/#).

## Task 4: Solve an identification task on testbench data 

In the previous part of the task, we generated the simulation data and presented the toolchain that can be used to solve the problem. In this subtask, measurement data from a double pendulum test rig is now read in so that the true length values are unknown to you in this subtask. Use the existing toolchain to estimate the pole lengths of the pendulum. Your specific solution path is not set in stone as we will only check for the found pendulum length values, which should deviate less than 5 % from the (unknown) ground truth values.

**Hint #1:** The search area can be limited to a range of $[0.1, 3.0]$ for both parameters.

**Hint #2:** You can compare the final loss value after training with the successful identification from the above subtasks to get an idea if the found results in task 4) are feasible or not.

In [31]:
import h5py
import numpy as np

# Open the MATLAB v7.3 file using h5py in read mode
with h5py.File("data_task_4.mat", 'r') as data_1:
    
    # Extract the datasets into numpy arrays and transpose them (.T) 
    # to match the (states, timepoints) shape expected by your objective function
    x_mes = np.array(data_1["x"]).T
    y_mes = np.array(data_1["y"]).T

# Verify the shapes are what you expect (e.g., (4, N) and (2, N))
print(f"Shape of x_mes: {x_mes.shape}")
print(f"Shape of y_mes: {y_mes.shape}")

Shape of x_mes: (4, 1001)
Shape of y_mes: (2, 1001)


In [32]:
# 1. Initialization
# Provide a reasonable initial guess within the [0.1, 3.0] bounds
p_init = np.array([1.5, 1.5]) 
tspan = (0.0, 1.0)
dt = 1e-3

# Extract the exact initial state from the first time-step of the real data
# This ensures our simulation starts exactly where the physical pendulum started
x0 = x_mes[:, 0] 

# 2. Solver Setup
# L-BFGS-B is the standard bounded gradient-descent solver in SciPy
solver = 'L-BFGS-B'
parameter_bounds = [(0.1, 3.0), (0.1, 3.0)]

### BEGIN SOLUTION
# Create the objective function wrapper, passing in the real-world y_mes and x0
objective_func = lambda p: simulation_objective(
    p, y_mes, double_pendulum_scipy, x0, tspan, dt
)

# Run the optimization
result = minimize(
    fun=objective_func,
    x0=p_init,
    method=solver,
    bounds=parameter_bounds,
    tol=1e-5
)

# Extract the final parameters
optimal_parameters = result.x
### END SOLUTION

print(f"Optimal Parameters: {optimal_parameters}")
print(f"Final MSE Loss: {result.fun}")

Optimal Parameters: [0.85037177 1.41823918]
Final MSE Loss: 0.01656113982927501


In [33]:
### BEGIN TESTS
# Verify we have exactly two parameters (l1 and l2)
assert optimal_parameters.shape == (2,), "Optimal parameters must have shape (2,)"

# Verify the identified lengths match the actual testbench lengths within a 5% tolerance
expected_lengths = np.array([0.85, 1.42])

assert np.allclose(optimal_parameters, expected_lengths, atol=5e-2, rtol=0), \
    f"Identified lengths {optimal_parameters} deviate too far from expected {expected_lengths}"

print("Test passed successfully. Pendulum lengths identified!")
### END TESTS

Test passed successfully. Pendulum lengths identified!
